# Coupled RAFT and DEQ-Flow in SILVA

The equilibrium state is $(h,u)$:

$$
h^+=\operatorname{ConvGRU}(h,c,m(u,C(u))),\qquad
u^+=u+\Delta_\theta(h^+).
$$

This package-native case exposes residual-encoder stages and stride,
correlation pyramid levels and radius, motion/GRU widths, global
aggregation, solver and gradient rules, sparse correction indices,
learned convex upsampling, fixed-point reuse, and custom encoder or
update modules.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [Path.cwd(), Path("/content/silva-networks"), Path("/content/drive/MyDrive/silva-networks")]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import torch

from silva_networks import (
    SILVARAFTDEQ,
    SolverConfig,
    make_silva_translation_flow_batch,
    silva_flow_fixed_point_correction_loss,
)

torch.manual_seed(13)
batch = make_silva_translation_flow_batch(
    batch_size=1, channels=1, height=8, width=8, shift=(1.0, 0.0)
)
config = SolverConfig(
    solver="picard",
    max_iter=3,
    alpha=0.5,
    indexing=(1, 2),
    backward_mode="implicit",
    backward_solver="gmres",
    backward_max_iter=8,
)
model = SILVARAFTDEQ(
    in_channels=1,
    feature_dim=8,
    hidden_dim=4,
    context_dim=4,
    encoder_channels=(4,),
    encoder_residual_blocks=1,
    encoder_dropout=0.0,
    output_stride=2,
    corr_levels=2,
    corr_radius=1,
    motion_dim=8,
    flow_head_dim=8,
    gru_kernel_size=3,
    correlation_hidden_dims=(8, 8),
    flow_hidden_dims=(8, 4),
    correction_steps=1,
    config=config,
)

## Solve, Sparse Corrections, and Exact Gradient

`indexing` stores selected numerical states. Short differentiable
corrections turn them into auxiliary flow predictions without
retaining the entire forward solver graph.

In [ ]:
result = model(batch.image1, batch.image2, return_result=True)
predictions = result.flow_sequence or [result.flow]
loss = silva_flow_fixed_point_correction_loss(
    predictions, batch.flow, valid=batch.valid, gamma=0.8
)
loss.backward()
finite_gradients = all(
    parameter.grad is None or torch.isfinite(parameter.grad).all()
    for parameter in model.parameters()
)
print("flow", result.flow.shape)
print("low-resolution state", result.low_resolution_flow.shape)
print("correction predictions", len(predictions))
print("finite gradients", bool(finite_gradients))

## Reuse the Fixed Point

A previous hidden/flow equilibrium can initialize a related image
pair. Whether this is appropriate across frames or augmentations is
an experiment choice.

In [ ]:
reused = model(batch.image1, batch.image2, cached_state=result.cached_state)
print("reused flow", reused.shape)

## Reproduction Boundary and Citations

This smoke run validates the coupled state, correlation/GRU update,
learned upsampling, correction loss, implicit backward path, and
reuse contract. Paper metrics require the source dataset mixtures,
augmentations, schedules, evaluation code, resolution, and model
dimensions.

Cite RAFT for all-pairs correlation and recurrent refinement,
DEQ-Flow for the equilibrium optical-flow formulation and sparse
correction/reuse strategy, and SILVA for this generalized package API.